# Lazy Frame & Logical Plan 🗺️

> The relational algebra tree and the user-facing `LazyFrame` API.

In [ ]:
#| default_exp lazy_frame

In [ ]:
#| export
from typing import Any, List, Literal, Optional, Union, Dict
from dataclasses import dataclass
from mxframe.lazy_expr import Expr, col
import pyarrow as pa


## Logical Plan Nodes 🧩

These classes represent the relational operations in our query plan.

In [ ]:
#| export
class LogicalPlan:
    """Base class for all logical plan nodes."""
    pass

@dataclass
class Scan(LogicalPlan):
    """Represents reading data from a source (e.g., a PyArrow Table)."""
    table: pa.Table

@dataclass
class Filter(LogicalPlan):
    """Represents filtering rows based on a predicate."""
    input: LogicalPlan
    predicate: Expr

@dataclass
class Project(LogicalPlan):
    """Represents selecting or computing columns."""
    input: LogicalPlan
    exprs: List[Expr]

@dataclass
class Aggregate(LogicalPlan):
    """Represents grouping and aggregating data."""
    input: LogicalPlan
    group_by: List[Expr]
    aggs: List[Expr]

## The `LazyFrame` API 🚀

This is the main entry point for users to build queries.

In [ ]:
#| export
# Device type used by compute()
DeviceType = Literal["cpu", "gpu", "auto"]


class LazyFrame:
    """A lazy DataFrame that builds a logical plan."""
    def __init__(self, plan):
        self.plan = plan

    def select(self, *exprs):
        """Select columns or compute new ones."""
        parsed_exprs = [col(e) if isinstance(e, str) else e for e in exprs]
        return LazyFrame(Project(self.plan, parsed_exprs))

    def filter(self, predicate):
        """Filter rows based on a condition."""
        return LazyFrame(Filter(self.plan, predicate))

    def groupby(self, *by):
        """Group data by one or more columns."""
        parsed_by = [col(b) if isinstance(b, str) else b for b in by]
        return LazyGroupBy(self, parsed_by)

    def compute(self, device: DeviceType = "cpu") -> pa.Table:
        """Execute the logical plan and return the result.

        Args:
            device: "cpu" (default), "gpu", or "auto".
                "cpu"  — always run on CPU.
                "gpu"  — run on GPU; raises RuntimeError if no GPU is available.
                "auto" — use GPU when available and N > 100K rows, else CPU.
        """
        from mxframe.custom_ops import CustomOpsCompiler
        return CustomOpsCompiler(device=device).compile_and_run(self.plan)


class LazyGroupBy:
    """Intermediate object for grouping operations."""
    def __init__(self, df, by):
        self.df = df
        self.by = by

    def agg(self, *aggs) -> LazyFrame:
        """Compute aggregations for each group."""
        return LazyFrame(Aggregate(self.df.plan, self.by, list(aggs)))


## Tests 🧪

Let's verify that our `LazyFrame` builds the correct `LogicalPlan`.

In [ ]:
import pyarrow as pa
from mxframe.lazy_expr import col, lit

# Dummy data
table = pa.table({'a': [1, 2, 3], 'b': [4, 5, 6]})
lf = LazyFrame(Scan(table))

# Test select
lf_sel = lf.select(col('a') + lit(1), 'b')
assert isinstance(lf_sel.plan, Project)
assert len(lf_sel.plan.exprs) == 2
assert lf_sel.plan.exprs[1].op == 'col'

# Test filter
lf_fil = lf.filter(col('a') > lit(1))
assert isinstance(lf_fil.plan, Filter)
assert lf_fil.plan.predicate.op == 'gt'

# Test groupby
lf_agg = lf.groupby('a').agg(col('b').sum())
assert isinstance(lf_agg.plan, Aggregate)
assert len(lf_agg.plan.group_by) == 1
assert len(lf_agg.plan.aggs) == 1
assert lf_agg.plan.aggs[0].op == 'sum'

print("All tests passed! 🎉")

All tests passed! 🎉
